# Phase 3: Fine-Tuning BGE Embedding Model (`BAAI/bge-base-en-v1.5`)

This notebook fine-tunes the dense retrieval embedding model on your enterprise dataset using **MultipleNegativesRankingLoss**.

### Features:
- Automatically finds and resumes from checkpoints if interrupted
- Checkpoints saved every 100 steps
- Evaluates on unseen queries
- Saves clean deployable adapter weights for Antigravity Enterprise RAG


In [ ]:
# 1. Install required packages
!pip install -q sentence-transformers datasets accelerate peft
!nvidia-smi


In [ ]:
# 2. Imports and Environment Setup
import os
import json
import glob
import torch
from pathlib import Path
from torch.utils.data import DataLoader
from sentence_transformers import SentenceTransformer, InputExample, losses, evaluation

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Training device: {DEVICE}')

# Paths on Kaggle
DATASET_DIR = '/kaggle/input/enterprise-rag-dataset'
CHECKPOINT_DIR = '/kaggle/working/checkpoints_embedding'
FINAL_OUTPUT_DIR = '/kaggle/working/embedding_adapter'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(FINAL_OUTPUT_DIR, exist_ok=True)


In [ ]:
# 3. Load Dataset (questions.jsonl and chunks.jsonl)
print('Loading corpus and training questions...')

# Map chunk_id/document_id to text
doc_text_map = {}
chunks_path = os.path.join(DATASET_DIR, 'chunks.jsonl')
if not os.path.exists(chunks_path):
    # Search recursively if placed in subdirectory
    found = glob.glob(f'{DATASET_DIR}/**/chunks.jsonl', recursive=True)
    if found: chunks_path = found[0]

with open(chunks_path, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line: continue
        c = json.loads(line)
        cid = c.get('chunk_id')
        did = c.get('document_id')
        txt = c.get('text', '')
        if cid: doc_text_map[cid] = txt
        if did and did not in doc_text_map: doc_text_map[did] = txt

print(f'Loaded {len(doc_text_map):,} document passages.')

# Load queries
questions_path = os.path.join(DATASET_DIR, 'questions.jsonl')
if not os.path.exists(questions_path):
    found = glob.glob(f'{DATASET_DIR}/**/questions.jsonl', recursive=True)
    if found: questions_path = found[0]

train_samples = []
eval_queries = {}
eval_corpus = {}
eval_relevant_docs = {}

with open(questions_path, 'r', encoding='utf-8') as f:
    lines = [json.loads(l.strip()) for l in f if l.strip()]

# 85% train, 15% eval split
split_idx = int(len(lines) * 0.85)
train_lines = lines[:split_idx]
eval_lines = lines[split_idx:]

for item in train_lines:
    q = item.get('question')
    doc_ids = item.get('expected_doc_ids', [])
    for did in doc_ids:
        if did in doc_text_map:
            pos_text = doc_text_map[did]
            train_samples.append(InputExample(texts=[q, pos_text]))

for i, item in enumerate(eval_lines):
    qid = f'q_{i}'
    q = item.get('question')
    eval_queries[qid] = q
    eval_relevant_docs[qid] = set()
    for did in item.get('expected_doc_ids', []):
        if did in doc_text_map:
            eval_corpus[did] = doc_text_map[did]
            eval_relevant_docs[qid].add(did)

print(f'Constructed {len(train_samples):,} positive query-passage training pairs.')
print(f'Constructed {len(eval_queries):,} evaluation queries.')


In [ ]:
# 4. Initialize Model and Check for Existing Checkpoints
BASE_MODEL_NAME = 'BAAI/bge-base-en-v1.5'

# Look for latest checkpoint or completed model to resume/continue
checkpoints = sorted(glob.glob(f'{CHECKPOINT_DIR}/checkpoint-*'), key=os.path.getmtime)
if checkpoints:
    latest_checkpoint = checkpoints[-1]
    print(f'🔄 Resuming from existing checkpoint: {latest_checkpoint}')
    model = SentenceTransformer(latest_checkpoint, device=DEVICE)
elif os.path.exists(FINAL_OUTPUT_DIR) and os.listdir(FINAL_OUTPUT_DIR):
    print(f'🔄 Resuming from completed model: {FINAL_OUTPUT_DIR}')
    model = SentenceTransformer(FINAL_OUTPUT_DIR, device=DEVICE)
else:
    print(f'Starting fresh training from base model: {BASE_MODEL_NAME}')
    model = SentenceTransformer(BASE_MODEL_NAME, device=DEVICE)


In [ ]:
# 5. Configure DataLoader and Loss
train_dataloader = DataLoader(train_samples, shuffle=True, batch_size=16)
train_loss = losses.MultipleNegativesRankingLoss(model)

# Evaluator
evaluator = evaluation.InformationRetrievalEvaluator(
    eval_queries, eval_corpus, eval_relevant_docs,
    name='eval_holdout',
    show_progress_bar=True,
    mrr_at_k=[5, 10],
    ndcg_at_k=[10]
) if eval_queries else None


In [ ]:
# 6. Execute Fine-Tuning with Checkpointing
EPOCHS = 4
WARMUP_STEPS = int(len(train_dataloader) * EPOCHS * 0.1)

print(f'Starting training for {EPOCHS} epochs ({len(train_dataloader)} steps per epoch)...')

model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    evaluator=evaluator,
    epochs=EPOCHS,
    warmup_steps=WARMUP_STEPS,
    output_path=CHECKPOINT_DIR,
    evaluation_steps=100,
    checkpoint_path=CHECKPOINT_DIR,
    checkpoint_save_steps=100,
    checkpoint_save_total_limit=2,
    optimizer_params={'lr': 2e-5},
    show_progress_bar=True
)

# Save final model
model.save(FINAL_OUTPUT_DIR)
print(f'✅ Model fine-tuning complete! Final model saved to: {FINAL_OUTPUT_DIR}')

# List output artifacts
!ls -lh /kaggle/working/embedding_adapter
